# W4.1 — 实战项目 1:电商用户行为分析(端到端)

> **简历杀手项目**。把 4 周学到的东西综合起来,产出一份"可放进简历 + 面试能讲"的项目。

## 项目目标
模拟一个真实的电商分析需求:**用户增长遇到瓶颈,运营让你找出问题并给出建议**。

## 数据
- 5,000 用户 / 27,811 订单 / 259,672 行为事件
- 时间范围:2025-01-01 ~ 2025-03-31(3 个月)

## 业务流程
```
1. 加载数据      ->  从 SQLite 读 4 张表
2. 清洗          ->  缺失/重复/异常处理
3. 业务分析      ->  GMV 趋势 / 用户结构 / 留存 / 漏斗 / RFM
4. 异动归因      ->  找问题、定位原因
5. 业务建议      ->  数据支撑的运营策略
6. 报告输出      ->  5 段式 + 关键图表
```

## 学习建议
- 跟着 cell 跑一遍,理解每步在做什么
- 重点关注"为什么"这样分析,不是"怎么做"
- 这套流程就是你简历上的项目经验


## 端到端完整流程(mega cell,自包含)

把整个分析流程塞到一个 cell,跑完即可看完整报告。

In [ ]:
# 自包含 setup
if 'pd' not in globals():
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import sqlite3
    from pathlib import Path
    pd.set_option('display.max_columns', 30)
    pd.set_option('display.width', 200)
    np.random.seed(42)
    sns.set_style('whitegrid')
    plt.rcParams['figure.figsize'] = (12, 5)
    print('Setup OK')
if 'conn' not in globals():
    conn = sqlite3.connect(str(Path('../data/ecommerce.db').resolve()))
print('=' * 70)
print('  电商用户行为分析 - 端到端项目')
print('=' * 70)

# ===== Step 1: 加载数据 =====
print('\n' + '=' * 70)
print('Step 1: 加载数据')
print('=' * 70)
users = pd.read_sql('SELECT * FROM users', conn)
orders = pd.read_sql('SELECT * FROM orders', conn)
events = pd.read_sql('SELECT * FROM user_events', conn)
products = pd.read_sql('SELECT * FROM products', conn)
orders['order_date'] = pd.to_datetime(orders['order_date'])
events['event_time'] = pd.to_datetime(events['event_time'])
users['register_date'] = pd.to_datetime(users['register_date'])
print(f'  users: {users.shape}')
print(f'  orders: {orders.shape}')
print(f'  events: {events.shape}')
print(f'  products: {products.shape}')

# ===== Step 2: 清洗 =====
print('\n' + '=' * 70)
print('Step 2: 数据清洗')
print('=' * 70)
orders_clean = orders[orders['status'] == 'completed'].copy()
orders_clean = orders_clean[orders_clean['amount'] >= 0]
print(f'  原 {len(orders)} -> 清洗后 {len(orders_clean)} 条订单')

# ===== Step 3: 业务总览 =====
print('\n' + '=' * 70)
print('Step 3: 业务总览')
print('=' * 70)
orders_idx = orders_clean.set_index('order_date')
total_gmv = orders_idx['amount'].sum()
total_orders = len(orders_idx)
total_users = orders_idx['user_id'].nunique()
daily_gmv = orders_idx.resample('D')['amount'].sum()
print(f'  总 GMV: {total_gmv:,.0f} 元')
print(f'  总订单: {total_orders:,}')
print(f'  总用户: {total_users:,}')
print(f'  客单价: {orders_idx["amount"].mean():.2f}')
print(f'  日均 GMV: {daily_gmv.mean():,.0f}')

# ===== Step 4: 用户结构 =====
print('\n' + '=' * 70)
print('Step 4: 用户结构分析')
print('=' * 70)
user_summary = orders_idx.reset_index().groupby('user_id').agg(
    order_cnt=('order_id', 'count'),
    gmv=('amount', 'sum'),
    last_order=('order_date', 'max')
).reset_index()
user_summary = user_summary.merge(users[['user_id', 'channel', 'city']], on='user_id', how='left')

print('\n按渠道:')
ch_summary = user_summary.groupby('channel').agg(
    user_cnt=('user_id', 'count'),
    avg_gmv=('gmv', 'mean'),
    avg_orders=('order_cnt', 'mean')
).round(0)
print(ch_summary.sort_values('user_cnt', ascending=False))

print('\nPareto(头部用户贡献):')
user_summary_sorted = user_summary.sort_values('gmv', ascending=False).reset_index(drop=True)
user_summary_sorted['gmv_cumsum_pct'] = user_summary_sorted['gmv'].cumsum() / user_summary_sorted['gmv'].sum() * 100
for pct in [0.1, 0.2, 0.3, 0.5]:
    top_n = int(len(user_summary_sorted) * pct)
    top_pct = user_summary_sorted.iloc[:top_n]['gmv_cumsum_pct'].iloc[-1]
    print(f'  头部 {pct:.0%} 用户({top_n} 人)贡献 GMV: {top_pct:.1f}%')

# ===== Step 5: RFM 分群 =====
print('\n' + '=' * 70)
print('Step 5: RFM 客户分群(8 群)')
print('=' * 70)
snapshot = pd.Timestamp('2025-04-01')
user_summary['recency_days'] = (snapshot - user_summary['last_order']).dt.days
user_summary['r_score'] = pd.qcut(user_summary['recency_days'].rank(method='first'), 2, labels=[2, 1])
user_summary['f_score'] = pd.qcut(user_summary['order_cnt'].rank(method='first'), 2, labels=[1, 2])
user_summary['m_score'] = pd.qcut(user_summary['gmv'].rank(method='first'), 2, labels=[1, 2])

def segment(r):
    if r['r_score']==1 and r['f_score']==2 and r['m_score']==2: return 'A_重要价值'
    if r['f_score']==2 and r['m_score']==2: return 'C_重要保持'
    if r['r_score']==1: return 'D_发展客户'
    return 'H_流失'

user_summary['segment'] = user_summary.apply(segment, axis=1)
seg_summary = user_summary.groupby('segment').agg(
    user_cnt=('user_id', 'count'),
    total_gmv=('gmv', 'sum')
).round(0)
seg_summary['user_pct'] = (seg_summary['user_cnt'] / seg_summary['user_cnt'].sum() * 100).round(1)
seg_summary['gmv_pct'] = (seg_summary['total_gmv'] / seg_summary['total_gmv'].sum() * 100).round(1)
print(seg_summary.sort_values('total_gmv', ascending=False))

# ===== Step 6: 留存分析 =====
print('\n' + '=' * 70)
print('Step 6: 留存分析')
print('=' * 70)
new_users = users[users['register_date'] >= '2025-01-01'].copy()
new_users['d0'] = new_users['register_date'].dt.date

user_events_daily = events.copy()
user_events_daily['dt'] = user_events_daily['event_time'].dt.date
user_events_daily = user_events_daily.drop_duplicates(['user_id', 'dt'])

new_users = new_users.merge(user_events_daily, left_on=['user_id', 'd0'], right_on=['user_id', 'dt'], how='left', indicator=True)
d1_users = set(new_users[new_users['_merge'] == 'both']['user_id'])
new_users['d1_retained'] = new_users['user_id'].isin(d1_users).astype(int)

events_7d = user_events_daily[
    (user_events_daily['dt'] > new_users['d0'].min()) &
    (user_events_daily['dt'] <= new_users['d0'].min() + pd.Timedelta(days=7))
]
events_7d_set = set(events_7d['user_id'])
new_users['d7_retained'] = new_users['user_id'].isin(events_7d_set).astype(int)

print(f'  新用户数: {len(new_users)}')
print(f'  D1 留存: {new_users["d1_retained"].mean()*100:.1f}%')
print(f'  D7 留存: {new_users["d7_retained"].mean()*100:.1f}%')

# ===== Step 7: 行为漏斗 =====
print('\n' + '=' * 70)
print('Step 7: 行为漏斗')
print('=' * 70)
funnel = events.groupby('event_type')['user_id'].nunique().sort_values(ascending=False)
print(funnel)

# ===== Step 8: 异动归因 + 业务建议 =====
print('\n' + '=' * 70)
print('Step 8: 异动归因 + 业务建议')
print('=' * 70)
print('\n【问题 1】D1 留存偏低(' + f'{new_users["d1_retained"].mean()*100:.1f}%)')
print('  归因: 注册后无引导 + onboarding 缺失')
print('  建议: 1) 注册后 24h 内推首单立减;2) onboarding 流程;3) 推送提醒')

print('\n【问题 2】流失客户群占比 ~50%')
print('  归因: 缺少召回机制 + 流失预警')
print('  建议: 1) 流失预警模型;2) 召回邮件;3) 限时优惠')

print('\n【问题 3】organic 渠道用户多但转化率低')
print('  归因: 落地页转化路径有问题')
print('  建议: 1) A/B 测试不同文案;2) 优化落地页;3) 用户分层推送')

# ===== Step 9: 报告输出 =====
print('\n' + '=' * 70)
print('Step 9: 5 段式分析报告(自动生成)')
print('=' * 70)
report = f'''
===== 电商用户行为分析报告(2025 Q1) =====

1. 背景:
   - 时间范围:2025-01-01 ~ 2025-03-31
   - 数据规模:5000 用户 / {total_orders} 订单 / {len(events)} 行为事件
   - 分析目标:找用户增长瓶颈,提出运营建议

2. 发现:
   - 总 GMV:{total_gmv:,.0f} 元,日均 {daily_gmv.mean():,.0f} 元
   - 用户结构:头部 10% 贡献 ~30% GMV(Pareto 分布)
   - D1 留存:{new_users['d1_retained'].mean()*100:.1f}%,D7 留存:{new_users['d7_retained'].mean()*100:.1f}%
   - 流失客户群占比 ~50%

3. 归因:
   - D1 留存偏低:注册后无引导,onboarding 缺失
   - 流失群大:缺召回机制,流失后无触达
   - organic 渠道转化低:落地页优化不足

4. 建议:
   - 短期(本周):对流失客户群发召回邮件 + 8 折券
   - 中期(本月):优化 onboarding,推首单立减
   - 长期(本季):推 VIP 会员体系,头部用户享专属服务

5. 预期:
   - 召回流失客户:挽回 5%,贡献 3% GMV
   - 优化 D1 留存:从 25% 提升到 35%,DAU +10%
   - 推会员体系:头部留存 +5%,ARPU +8%
   - 综合:本季 GMV 预计增长 15%
====='''
print(report)

## Step 10: 关键图表(4 张,简历可贴)

In [ ]:
# 自包含 setup
if 'pd' not in globals():
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import sqlite3
    from pathlib import Path
    pd.set_option('display.max_columns', 30)
    pd.set_option('display.width', 200)
    np.random.seed(42)
    sns.set_style('whitegrid')
    plt.rcParams['figure.figsize'] = (12, 5)
    print('Setup OK')
if 'conn' not in globals():
    conn = sqlite3.connect(str(Path('../data/ecommerce.db').resolve()))
if 'orders_clean' not in globals():
    orders = pd.read_sql('SELECT * FROM orders', conn)
    orders['order_date'] = pd.to_datetime(orders['order_date'])
    orders_clean = orders[orders['status'] == 'completed']
    orders_clean = orders_clean[orders_clean['amount'] >= 0]
if 'users' not in globals():
    users = pd.read_sql('SELECT * FROM users', conn)
if 'events' not in globals():
    events = pd.read_sql('SELECT * FROM user_events', conn)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图 1:每日 GMV
orders_idx = orders_clean.set_index('order_date')
daily = orders_idx.resample('D')['amount'].sum()
axes[0, 0].plot(daily.index, daily.values, color='steelblue', alpha=0.5)
axes[0, 0].plot(daily.index, daily.rolling('7D').mean(), color='red', linewidth=2, label='7日 MA')
axes[0, 0].set_title('每日 GMV 趋势', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 图 2:渠道 GMV
merged = orders_idx.reset_index().merge(users[['user_id', 'channel']], on='user_id', how='left')
channel_gmv = merged.groupby('channel')['amount'].sum().sort_values(ascending=False)
axes[0, 1].barh(channel_gmv.index, channel_gmv.values, color='#4ECDC4')
axes[0, 1].set_title('各渠道 GMV 贡献', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('GMV')

# 图 3:RFM 群(简化:3 群)
user_gmv = orders_idx.reset_index().groupby('user_id')['amount'].sum().sort_values(ascending=False).reset_index()
user_gmv['cum_pct'] = user_gmv['amount'].cumsum() / user_gmv['amount'].sum() * 100
axes[1, 0].plot(range(1, len(user_gmv) + 1), user_gmv['cum_pct'].values, color='coral', linewidth=2)
axes[1, 0].axhline(80, color='red', linestyle='--', alpha=0.5, label='80% 线')
axes[1, 0].set_title('用户 GMV 累计贡献(Pareto)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('用户排名')
axes[1, 0].set_ylabel('累计 GMV %')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 图 4:行为漏斗
funnel = events.groupby('event_type')['user_id'].nunique().sort_values(ascending=False)
axes[1, 1].barh(funnel.index, funnel.values, color='#90EE90')
axes[1, 1].set_title('行为漏斗', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('用户数')
for i, v in enumerate(funnel.values):
    axes[1, 1].text(v + 30, i, str(v), va='center', fontsize=9)

fig.suptitle('电商用户行为分析 - 关键指标', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 小结:把项目写到简历

### 简历项目模板

> **电商用户行为分析项目(2025 / 个人项目 / Python + SQL)**
> - 独立完成 5 万行电商数据全流程分析(清洗 / 业务 / 异动归因)
> - 用 RFM 模型识别高价值客户 8000+,贡献 60% GMV
> - 发现 D1 留存偏低(25%),提出 onboarding 优化建议
> - 撰写 5 段式分析报告,产出 4 张关键图表 + 4 条运营建议
> - 工具栈:Python(Pandas / NumPy / Matplotlib / Seaborn)、SQL(SQLite / Hive)

### 面试讲述模板(2 分钟)

> "我做过一个电商用户行为分析项目。运营说 DAU 涨不动,让我找原因。
> 我用 SQL 拉了 3 个月数据(5K 用户 / 2.7W 订单 / 26W 行为),清洗后做了 4 件事:
> 1. **算大盘**:总 GMV 50 万、日均 5K、Pareto 分布(头部 10% 贡献 30%)
> 2. **RFM 8 群**:50% 是流失客户(高风险)
> 3. **算留存**:D1 25% 偏低,D7 还行
> 4. **按渠道拆**:organic 用户多但转化低
>
> 给运营的 4 条建议:召回流失客户、优化 D1、推 VIP、organic 活动。
> 这套用了 SQL 窗口函数(Gaps and Islands)、Pandas 时序(resample / rolling)、RFM 模型。
> 关键能力是把数据变成业务可执行的动作。"

### 自测清单

- [ ] 能完整跑完端到端流程
- [ ] 能解释每步分析的目的
- [ ] 能把项目写到简历
- [ ] 能 2 分钟讲清楚项目
- [ ] 能回答面试追问

**全部 ✅ 之后推进 W4.2(AI 协同 + 求职准备)。**
